In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, roc_curve, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import shap
import seaborn as sns
import warnings
import math
warnings.filterwarnings('ignore')
from dagma.linear import DagmaLinear

In [2]:
from causalnex.structure.notears import from_pandas

In [3]:
from sklearn.preprocessing import MinMaxScaler

In [4]:
from typing import Tuple

In [5]:
df = pd.read_excel("Telco-Customer-Churn.xlsx")

In [6]:
import pickle
import copy
from causalnex.network import BayesianNetwork
from causalnex.structure import StructureModel
from causalnex.plots import plot_structure
from IPython.display import Image
from sklearn.metrics import mutual_info_score
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE

In [7]:
import json

In [8]:
import networkx as nx

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        7043 non-null   object 
 1   Gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   Tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [10]:
df.describe()

,SeniorCitizen,Tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [459]:
# TOTAL CHARGES TO NUMERIC
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [460]:
# MonthlyCharges TO NUMERIC
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df['MonthlyCharges'].fillna(df['MonthlyCharges'].median(), inplace=True)

In [461]:
# Tenure TO NUMERIC
df['Tenure'] = pd.to_numeric(df['Tenure'], errors='coerce')
df['Tenure'].fillna(df['Tenure'].median(), inplace=True)

In [462]:
# DROP CUSTOMER ID
# df.drop(['CustomerID'], axis=1, inplace=True)

In [463]:
# DEFINE CATEGORICAL AND NUMERICAL
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [464]:
print (categorical_cols)

Index(['CustomerID', 'Gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='object')


In [465]:
print (numerical_cols)

Index(['SeniorCitizen', 'Tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')


In [466]:
df[categorical_cols].nunique()  

CustomerID          7043
Gender                 2
Partner                2
Dependents             2
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
Churn                  2
dtype: int64

In [467]:
# RENAME CATEGORIES TO AVOID SPACES
for columna in categorical_cols:
    df[columna] = df[columna].astype(str).str.replace(' ', '_')

In [468]:
# ENCODE TARGET VARIABLE
df['Churn'] = df['Churn'].replace({'No': 0, 'Yes': 1})

In [469]:
# -----------------------------
# Binary categorical columns
# -----------------------------
binary_cols = [
    'Gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

# -----------------------------
# Multi-category columns
# -----------------------------
multi_cat_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

# -----------------------------
# Numerical columns
# SeniorCitizen is binary (0,1)
# -----------------------------
numerical_cols = [
    'Tenure',
    'MonthlyCharges',
    'TotalCharges'
]

In [470]:
binary_mappings = {
    'Gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1}
}

In [471]:
for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping)


In [472]:
# ============================================
# OneHotEncoding for multi-category variables
# ============================================

ohe = OneHotEncoder(
    sparse_output=False,
    drop=None  # keep all categories for causal discovery
)

In [473]:
encoded_cat = ohe.fit_transform(df[multi_cat_cols])

In [474]:
print (encoded_cat)

[[0. 1. 0. ... 0. 1. 0.]
 [1. 0. 0. ... 0. 0. 1.]
 [1. 0. 0. ... 0. 0. 1.]
 ...
 [0. 1. 0. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 0. 1.]
 [1. 0. 0. ... 0. 0. 0.]]


In [475]:
#std = StandardScaler()

In [476]:
#encoded_num = std.fit_transform(df[numerical_cols])

In [477]:
minmax = MinMaxScaler(feature_range=(0, 1))

In [478]:
encoded_num = minmax.fit_transform(df[numerical_cols])

In [479]:
print (encoded_num)

[[0.01388889 0.11542289 0.0012751 ]
 [0.47222222 0.38507463 0.21586661]
 [0.02777778 0.35422886 0.01031041]
 ...
 [0.15277778 0.11293532 0.03780868]
 [0.05555556 0.55870647 0.03321025]
 [0.91666667 0.86965174 0.78764136]]


In [480]:
# Get encoded categorical feature names
columns = ohe.get_feature_names_out(multi_cat_cols)

In [481]:
print (columns)

['MultipleLines_No' 'MultipleLines_No_phone_service' 'MultipleLines_Yes'
 'InternetService_DSL' 'InternetService_Fiber_optic' 'InternetService_No'
 'OnlineSecurity_No' 'OnlineSecurity_No_internet_service'
 'OnlineSecurity_Yes' 'OnlineBackup_No' 'OnlineBackup_No_internet_service'
 'OnlineBackup_Yes' 'DeviceProtection_No'
 'DeviceProtection_No_internet_service' 'DeviceProtection_Yes'
 'TechSupport_No' 'TechSupport_No_internet_service' 'TechSupport_Yes'
 'StreamingTV_No' 'StreamingTV_No_internet_service' 'StreamingTV_Yes'
 'StreamingMovies_No' 'StreamingMovies_No_internet_service'
 'StreamingMovies_Yes' 'Contract_Month-to-month' 'Contract_One_year'
 'Contract_Two_year' 'PaymentMethod_Bank_transfer_(automatic)'
 'PaymentMethod_Credit_card_(automatic)' 'PaymentMethod_Electronic_check'
 'PaymentMethod_Mailed_check']


In [482]:
encoded_cat_df = pd.DataFrame(
    encoded_cat,
    columns=ohe.get_feature_names_out(multi_cat_cols),
    index=df.index
)

In [483]:
encoded_num_df = pd.DataFrame(
    encoded_num,
    columns=numerical_cols,
    index=df.index
)

In [484]:
# ============================================
# Final dataset
# ============================================

df_transformed = pd.concat(
    [
        df[binary_cols],
        encoded_num_df,
        encoded_cat_df,
        df[['SeniorCitizen','CustomerID','Churn']]
    ],
    axis=1
)

In [485]:
print (df_transformed.columns)

Index(['Gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
       'Tenure', 'MonthlyCharges', 'TotalCharges', 'MultipleLines_No',
       'MultipleLines_No_phone_service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber_optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No_internet_service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No_internet_service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No_internet_service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No_internet_service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No_internet_service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No_internet_service',
       'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One_year',
       'Contract_Two_year', 'PaymentMethod_Bank_transfer_(automatic)',
       'PaymentMethod_Credit_card_(automatic)',
     

In [486]:
df_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 42 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Gender                                   7043 non-null   int64  
 1   Partner                                  7043 non-null   int64  
 2   Dependents                               7043 non-null   int64  
 3   PhoneService                             7043 non-null   int64  
 4   PaperlessBilling                         7043 non-null   int64  
 5   Tenure                                   7043 non-null   float64
 6   MonthlyCharges                           7043 non-null   float64
 7   TotalCharges                             7043 non-null   float64
 8   MultipleLines_No                         7043 non-null   float64
 9   MultipleLines_No_phone_service           7043 non-null   float64
 10  MultipleLines_Yes                        7043 no

In [487]:
df_transformed.describe()

,Gender,Partner,Dependents,PhoneService,PaperlessBilling,Tenure,MonthlyCharges,TotalCharges,MultipleLines_No,MultipleLines_No_phone_service,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One_year,Contract_Two_year,PaymentMethod_Bank_transfer_(automatic),PaymentMethod_Credit_card_(automatic),PaymentMethod_Electronic_check,PaymentMethod_Mailed_check,SeniorCitizen,Churn
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,...,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.504756,0.483033,0.299588,0.903166,0.592219,0.449599,0.462803,0.261149,0.481329,0.096834,...,0.387903,0.550192,0.209144,0.240664,0.219225,0.216101,0.335794,0.228880,0.162147,0.265370
std,0.500013,0.499748,0.458110,0.295752,0.491457,0.341104,0.299403,0.261397,0.499687,0.295752,...,0.487307,0.497510,0.406726,0.427517,0.413751,0.411613,0.472301,0.420141,0.368612,0.441561
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,0.125000,0.171642,0.044245,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,1.000000,0.402778,0.518408,0.159090,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,0.763889,0.712438,0.434780,1.000000,0.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [488]:
print(df_transformed.head(10))

   Gender  Partner  Dependents  PhoneService  PaperlessBilling    Tenure  \
0       0        1           0             0                 1  0.013889   
1       1        0           0             1                 0  0.472222   
2       1        0           0             1                 1  0.027778   
3       1        0           0             0                 0  0.625000   
4       0        0           0             1                 1  0.027778   
5       0        0           0             1                 1  0.111111   
6       1        0           1             1                 1  0.305556   
7       0        0           0             0                 0  0.138889   
8       0        1           0             1                 1  0.388889   
9       1        0           1             1                 0  0.861111   

   MonthlyCharges  TotalCharges  MultipleLines_No  \
0        0.115423      0.001275               0.0   
1        0.385075      0.215867               1.0   
2   

In [489]:
#df_transformed.to_excel("Telco-Churn-OneHot-MinMax.xlsx", index=False)

In [11]:
#START 
df_transformed = pd.read_excel("Telco-Churn-OneHot-MinMax.xlsx")

In [12]:
# DROP CUSTOMER ID
df_transformed.drop(['CustomerID'], axis=1, inplace=True)

In [13]:
df_transformed = df_transformed.astype(float)

In [14]:
df_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 32 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Gender                                7043 non-null   float64
 1   Partner                               7043 non-null   float64
 2   Dependents                            7043 non-null   float64
 3   PhoneService                          7043 non-null   float64
 4   PaperlessBilling                      7043 non-null   float64
 5   Tenure                                7043 non-null   float64
 6   MonthlyCharges                        7043 non-null   float64
 7   TotalCharges                          7043 non-null   float64
 8   MultipleLines_No_phone_service        7043 non-null   float64
 9   MultipleLines_Yes                     7043 non-null   float64
 10  InternetService_Fiber_optic           7043 non-null   float64
 11  InternetService_N

In [149]:
def dagma_to_structure_model(
    df: pd.DataFrame,
    lambda1: float = 0.01,
    threshold: float = 0.05,
    #loss_type: str = "l2",
    loss_type: str = "logistic",
) -> Tuple[StructureModel, pd.DataFrame]:
    """
    Learn a DAG with DAGMA and convert it to a causalnex StructureModel.

    Returns:
        sm: causalnex StructureModel
        W_df: weighted adjacency matrix as DataFrame
    """

    # DAGMA expects numeric data
    df_num = df.select_dtypes(include=[np.number]).copy()
    
    df_num.info()
    
    # Standardization is recommended for continuous variables
    #X = StandardScaler().fit_transform(df_num.values)
    X = df_num.values
    
    print ("X:")
    print (X[0])

    # Learn weighted adjacency matrix
    model = DagmaLinear(loss_type=loss_type)
    W = model.fit(X, lambda1=lambda1)

    columns = df_num.columns.tolist()
    W_df = pd.DataFrame(W, index=columns, columns=columns)

    # Convert to StructureModel
    sm = StructureModel()

    for source in columns:
        sm.add_node(source)

    for i, source in enumerate(columns):
        for j, target in enumerate(columns):
            weight = W[i, j]

            if i != j and abs(weight) >= threshold:
                sm.add_edge(source, target, weight=float(weight))

    return sm, W_df


In [150]:
#DAGMA
#sm, W_df = dagma_to_structure_model(
#    df_transformed,
#    #lambda1=0.0001,
#    #threshold=0.0001
#    lambda1=0.001,
#    threshold=0.005
#)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 32 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Gender                                7043 non-null   float64
 1   Partner                               7043 non-null   float64
 2   Dependents                            7043 non-null   float64
 3   PhoneService                          7043 non-null   float64
 4   PaperlessBilling                      7043 non-null   float64
 5   Tenure                                7043 non-null   float64
 6   MonthlyCharges                        7043 non-null   float64
 7   TotalCharges                          7043 non-null   float64
 8   MultipleLines_No_phone_service        7043 non-null   float64
 9   MultipleLines_Yes                     7043 non-null   float64
 10  InternetService_Fiber_optic           7043 non-null   float64
 11  InternetService_N

  0%|          | 0/180000.0 [00:00<?, ?it/s]

In [ ]:
print("Edges:")
print(list(sm.edges(data=True)))

print("\nWeighted adjacency matrix:")
print(W_df.round(3))

In [155]:
#with open("model_dagma.pkl", "wb") as f:
#     pickle.dump(sm, f)

In [15]:
with open("model_dagma.pkl", "rb") as f:
    restored_model = pickle.load(f)

In [16]:
sm = restored_model

In [17]:
num_nodes = len(sm.nodes)
print(f"Number of nodes: {num_nodes}")

Number of nodes: 32


In [18]:
weights = []
all_edges = []

for u, v, data in sm.edges(data=True):
    w = (data['weight'])
    weights.append(w)
    all_edges.append({"From": u, "To": v, "Weight": round(w, 3)})

In [19]:
stats = {
    "Statistic": ["Count", "Mean", "Median", "Std Dev", "Min", "Max"],
    "Value": [
        len(weights),
        round(pd.Series(weights).mean(), 3),
        round(pd.Series(weights).median(), 3),
        round(pd.Series(weights).std(), 3),
        round(pd.Series(weights).min(), 3),
        round(pd.Series(weights).max(), 3),
    ]
}
df_stats = pd.DataFrame(stats)

In [20]:
print(df_stats)

  Statistic    Value
0     Count  275.000
1      Mean   -0.404
2    Median   -0.652
3   Std Dev    1.563
4       Min   -3.625
5       Max    3.932


In [21]:
sm_reduced = copy.deepcopy(restored_model)

In [22]:
threshold = 0.305
edges_to_remove = [(u, v) for u, v, w in sm_reduced.edges(data=True) if abs(w["weight"]) < threshold]
sm_reduced.remove_edges_from(edges_to_remove)

In [23]:
weights = []
reduced_edges = []

for u, v, data in sm_reduced.edges(data=True):
    w = data["weight"]
    weights.append(w)
    reduced_edges.append({"From": u, "To": v, "Weight": round(w, 3)})

In [24]:
stats = {
    "Statistic": ["Count", "Mean", "Median", "Std Dev", "Min", "Max"],
    "Value": [
        len(weights),
        round(pd.Series(weights).mean(), 3),
        round(pd.Series(weights).median(), 3),
        round(pd.Series(weights).std(), 3),
        round(pd.Series(weights).min(), 3),
        round(pd.Series(weights).max(), 3),
    ]
}
df_stats = pd.DataFrame(stats)

In [25]:
print (df_stats)

  Statistic    Value
0     Count  275.000
1      Mean   -0.404
2    Median   -0.652
3   Std Dev    1.563
4       Min   -3.625
5       Max    3.932


In [26]:
all_edge_attributes = {
    "color":{
        "color": "#FFFFFFD9",
    },
    "font": {
            "color": 'green',
            "face" : 'arial',
            "background": '#FFFFFFD9',
            "strokeWidth": 1 ,
            "strokeColor": 'green',
            "bold": True,
            "size": 30
        },
    "length": 300,
    "label" : ""
}

In [27]:
edge_attributes = {};
for u, v, data in sm_reduced.edges(data=True):
    weight = data["weight"]
    edge_attributes[(u, v)] = {
        "color": {
            "color": "black",
            "highlight": "#79d279"
        },
        "length": 500,
        "shadow": {
            "enabled": True,
            "color": "gray"
        },
        "label": ""  # Using the edge weight as the label
    }

In [28]:
all_node_attributes = {
    "font": {
        "color": "blue",
        "face": "Helvetica",
        "size": 40
    }
}

In [29]:
plot_options = {
    "bgcolor": "white"
}

In [30]:
viz = plot_structure(
    sm_reduced, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
#viz.toggle_physics(False)
viz.show("supporting_files/fully_connected_dagma.html")

supporting_files/fully_connected_dagma.html


In [31]:
center_node = "Churn"
subgraph = nx.ego_graph(sm_reduced, center_node, radius=1, center=True, undirected = True)

In [33]:
# Remove all isolated nodes
isolated_nodes = list(nx.isolates(subgraph))
subgraph.remove_nodes_from(isolated_nodes)

In [34]:
all_edges = [(u, v, w) for u, v, w in subgraph.edges(data=True)]

In [35]:
print (all_edges)

[('OnlineSecurity_No_internet_service', 'InternetService_No', {'origin': 'unknown', 'weight': 1.296792862260917}), ('PaymentMethod_Bank_transfer', 'PhoneService', {'origin': 'unknown', 'weight': 0.5729721300857193}), ('PaymentMethod_Bank_transfer', 'InternetService_Fiber_optic', {'origin': 'unknown', 'weight': -0.5862613867416129}), ('PaymentMethod_Bank_transfer', 'Churn', {'origin': 'unknown', 'weight': -0.4805701509421965}), ('PaymentMethod_Credit_card', 'PhoneService', {'origin': 'unknown', 'weight': 0.5975145323947655}), ('PaymentMethod_Credit_card', 'InternetService_Fiber_optic', {'origin': 'unknown', 'weight': -0.9511112112589881}), ('PaymentMethod_Credit_card', 'PaymentMethod_Bank_transfer', {'origin': 'unknown', 'weight': -2.9553548391310334}), ('PaymentMethod_Credit_card', 'Churn', {'origin': 'unknown', 'weight': -0.5778712808035411}), ('InternetService_Fiber_optic', 'PhoneService', {'origin': 'unknown', 'weight': 2.0977610967841125}), ('InternetService_Fiber_optic', 'Internet

In [36]:
edge_attributes = {};
for u, v, data in subgraph.edges(data=True):
    weight = data["weight"]
    edge_attributes[(u, v)] = {
        "color": {
            "color": "black",
            "highlight": "#79d279"
        },
        "length": 500,
        "shadow": {
            "enabled": True,
            "color": "gray"
        },
        "label": f"{weight:.3f}"  # Using the edge weight as the label
    }


In [37]:
opt = {
    "physics": {
        "enabled":False,
        "solver": "barnesHut",
        "barnesHut": {
            "gravitationalConstant": -2000,
            "centralGravity": 0.3,
            "springLength": 100,
            "springConstant": 0.04,
            "damping": 0.09,
            "avoidOverlap": 1
        }
    },
    "edges": {
        "smooth": {
            "enabled": True,
            "type": "cubicBezier",
            "forceDirection": "horizontal",
            "roundness": 0.5
        },
        "color": {
            "inherit": False
        },
        "arrows": {
            "to": {"enabled": True, "scaleFactor": 0.5}
        }
    },
    "interaction": {
        "hover": True,
        "navigationButtons": True
    },
    "layout": {
        "improvedLayout": True
    }
}

In [ ]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_dagma.html')

In [39]:
# Remove all outbound edges from center_node
out_edges = list(subgraph.out_edges(center_node))
subgraph.remove_edges_from(out_edges)

In [ ]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_dagma.html')

In [41]:
node = "Churn"
edges = []

In [42]:
# edges entering in Churn: X -> Churn
for source, target, data in subgraph.in_edges(node, data=True):
    edges.append({
        "Source": source,
        "Target": target,
        "Direction": "IN",
        "Weight": data.get("weight")
    })

In [43]:
df_edges = pd.DataFrame(edges)

df_edges.to_excel(
    "Global_Score_DAGMA.xlsx",
    index=False
)

In [44]:
from pathlib import Path

In [48]:
center_node = "Churn"
lambdas = [0.25, 0.50, 0.75]
output_file = Path("Path_Based_Structural_Importance_DAGMA.xlsx")

In [49]:
# ---------------------------------------------------------
# 1. Keep only direct parents of Churn
# ---------------------------------------------------------

parents = list(subgraph.predecessors(center_node))

nodes_to_keep = parents + [center_node]

In [50]:
# Induced subgraph:
# keeps Churn, its direct parents, AND all edges among parents
subgraph = subgraph.subgraph(nodes_to_keep).copy()

print(f"Number of direct parents of {center_node}: {len(parents)}")
print(f"Nodes in subgraph: {subgraph.number_of_nodes()}")
print(f"Edges in subgraph: {subgraph.number_of_edges()}")


Number of direct parents of Churn: 8
Nodes in subgraph: 9
Edges in subgraph: 31


In [51]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_dagma_pos_only_parents.html')

supporting_files/Churn_dagma_pos_only_parents.html


In [52]:
# ============================================================
# 2. HELPER FUNCTION
# ============================================================

def get_weight(G, source, target):
    return float(G[source][target].get("weight", 0.0))

In [53]:
# ============================================================
# 3. DIRECT COMPONENT D_i
# ============================================================
#
# Signed:
#     D_i = w(X_i -> Churn)
#
# Absolute:
#     |D_i|
#
# ============================================================

D_signed = {}
D_abs = {}

for node in parents:

    w = get_weight(
        subgraph,
        node,
        center_node
    )

    D_signed[node] = w
    D_abs[node] = abs(w)


In [54]:
# ============================================================
# 4. INDIRECT COMPONENT I_i
# ============================================================
#
# Only paths of EXACTLY length 2:
#
#     X_i -> X_j -> Churn
#
# Signed:
#
#     I_i_signed =
#         sum_j w_ij * w_j,Churn
#
# Absolute:
#
#     I_i_abs =
#         sum_j |w_ij * w_j,Churn|
#
# ============================================================

I_signed = {}
I_abs = {}

path_rows = []

for node_i in parents:

    signed_sum = 0.0
    abs_sum = 0.0

    for node_j in parents:

        if node_i == node_j:
            continue

        # Check for:
        #
        # node_i -> node_j -> Churn

        if (
            subgraph.has_edge(node_i, node_j)
            and
            subgraph.has_edge(node_j, center_node)
        ):

            w_ij = get_weight(
                subgraph,
                node_i,
                node_j
            )

            w_j_churn = get_weight(
                subgraph,
                node_j,
                center_node
            )

            path_signed = (
                w_ij * w_j_churn
            )

            path_abs = abs(path_signed)

            signed_sum += path_signed
            abs_sum += path_abs

            path_rows.append({
                "Source": node_i,
                "Via": node_j,
                "Target": center_node,

                "Weight_Source_Via": w_ij,
                "Weight_Via_Churn": w_j_churn,

                "Path_Signed": path_signed,
                "Path_Abs": path_abs
            })

    I_signed[node_i] = signed_sum
    I_abs[node_i] = abs_sum

In [55]:
# ============================================================
# 5. NORMALIZE D_i
# ============================================================
#
# D_abs_norm      in [0, 1]
# D_signed_norm   in [-1, 1]
#
# Both use:
#
#     max |D_i|
#
# ============================================================

max_D_abs = (
    max(D_abs.values())
    if D_abs
    else 0.0
)

D_abs_norm = {}
D_signed_norm = {}

for node in parents:

    if max_D_abs > 0:

        D_abs_norm[node] = (
            D_abs[node] / max_D_abs
        )

        D_signed_norm[node] = (
            D_signed[node] / max_D_abs
        )

    else:

        D_abs_norm[node] = 0.0
        D_signed_norm[node] = 0.0


In [56]:
# ============================================================
# 6. NORMALIZE I_i
# ============================================================
#
# Importance:
#     I_abs_norm in [0,1]
#
# Direction:
#     I_signed_norm in [-1,1]
#
# ============================================================

max_I_abs = (
    max(I_abs.values())
    if I_abs
    else 0.0
)

max_I_signed_abs = (
    max(abs(value) for value in I_signed.values())
    if I_signed
    else 0.0
)


I_abs_norm = {}
I_signed_norm = {}

for node in parents:

    if max_I_abs > 0:

        I_abs_norm[node] = (
            I_abs[node]
            / max_I_abs
        )

    else:

        I_abs_norm[node] = 0.0


    if max_I_signed_abs > 0:

        I_signed_norm[node] = (
            I_signed[node]
            / max_I_signed_abs
        )

    else:

        I_signed_norm[node] = 0.0


In [57]:
# ============================================================
# 7. CALCULATE S_i(lambda) AND C_i(lambda)
# ============================================================
#
# Structural importance:
#
# S_i(lambda) =
#     D_abs_norm
#     +
#     lambda * I_abs_norm
#
#
# Structural direction:
#
# C_i(lambda) =
#     D_signed_norm
#     +
#     lambda * I_signed_norm
#
# ============================================================

rows = []

for node in parents:

    row = {

        "Node": node,

        # Direct component
        "D_signed": D_signed[node],
        "D_abs": D_abs[node],

        "D_signed_norm": D_signed_norm[node],
        "D_abs_norm": D_abs_norm[node],

        # Indirect component
        "I_signed": I_signed[node],
        "I_abs": I_abs[node],

        "I_signed_norm": I_signed_norm[node],
        "I_abs_norm": I_abs_norm[node],
    }


    for lam in lambdas:

        # ----------------------------------------
        # IMPORTANCE
        # ----------------------------------------

        S = (
            D_abs_norm[node]
            +
            lam * I_abs_norm[node]
        )


        # ----------------------------------------
        # DIRECTION
        # ----------------------------------------

        C = (
            D_signed_norm[node]
            +
            lam * I_signed_norm[node]
        )


        row[f"S_{lam:.2f}"] = S
        row[f"C_{lam:.2f}"] = C


        # Direction of recommended intervention
        if C > 0:
            direction = "+"
        elif C < 0:
            direction = "-"
        else:
            direction = "0"

        row[f"Direction_{lam:.2f}"] = direction


    rows.append(row)


results = pd.DataFrame(rows)


In [58]:
# ============================================================
# 8. RANK S_i FOR EACH LAMBDA
# ============================================================
#
# Rank:
#     1 = highest structural importance
#
# Normalized rank:
#
#     1 = best
#     0 = worst
#
# NR = 1 - (Rank - 1)/(N - 1)
#
# ============================================================

n = len(results)

for lam in lambdas:

    S_col = f"S_{lam:.2f}"

    rank_col = f"Rank_{lam:.2f}"

    norm_rank_col = (
        f"Normalized_Rank_{lam:.2f}"
    )


    # Raw rank
    results[rank_col] = (
        results[S_col]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )


    # Normalized rank
    if n > 1:

        results[norm_rank_col] = (
            1
            -
            (
                results[rank_col] - 1
            )
            /
            (n - 1)
        )

    else:

        results[norm_rank_col] = 1.0



In [59]:
# ============================================================
# 9. SORT FOR DISPLAY
# ============================================================
#
# lambda = 0.50 used ONLY for display order.
# It does not affect calculations for other lambdas.
#
# ============================================================

results = (
    results
    .sort_values(
        "Rank_0.50"
    )
    .reset_index(drop=True)
)


In [60]:
# ============================================================
# 10. TABLE WITH INDIVIDUAL LENGTH-2 PATHS
# ============================================================

paths_df = pd.DataFrame(path_rows)

if not paths_df.empty:

    paths_df = (
        paths_df
        .sort_values(
            "Path_Abs",
            ascending=False
        )
        .reset_index(drop=True)
    )


In [61]:
# ============================================================
# 11. TABLE WITH ALL EDGES OF REDUCED SUBGRAPH
# ============================================================

edge_rows = []

for source, target, data in subgraph.edges(data=True):

    edge_rows.append({

        "Source": source,
        "Target": target,

        "Weight": data.get(
            "weight",
            np.nan
        ),

        "Direct_To_Churn":
            target == center_node
    })


edges_df = pd.DataFrame(edge_rows)


In [62]:
# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

display(results)

print("\nLength-2 paths:")
display(paths_df)

print("\nEdges in Churn subgraph:")
display(edges_df)

,Node,D_signed,D_abs,D_signed_norm,D_abs_norm,I_signed,I_abs,I_signed_norm,I_abs_norm,S_0.25,...,Direction_0.50,S_0.75,C_0.75,Direction_0.75,Rank_0.25,Normalized_Rank_0.25,Rank_0.50,Normalized_Rank_0.50,Rank_0.75,Normalized_Rank_0.75
0,Tenure,-2.474307,2.474307,-1.000000,1.000000,-4.574206,5.623476,-0.908286,0.736869,1.184217,...,-,1.552652,-1.681214,-,1,1.000000,1,1.000000,1,1.000000
1,MonthlyCharges,0.579521,0.579521,0.234215,0.234215,4.473271,7.631577,0.888243,1.000000,0.484215,...,+,0.984215,0.900398,+,3,0.714286,2,0.857143,2,0.857143
2,PaymentMethod_Mailed_check,-0.646082,0.646082,-0.261116,0.261116,0.026370,6.585471,0.005236,0.862924,0.476847,...,-,0.908309,-0.257189,-,4,0.571429,3,0.714286,3,0.714286
3,Contract_Month-to-month,0.685650,0.685650,0.277108,0.277108,5.036088,5.696705,1.000000,0.746465,0.463724,...,+,0.836957,1.027108,+,5,0.428571,4,0.571429,4,0.571429
4,InternetService_Fiber_optic,1.082427,1.082427,0.437467,0.437467,-2.247971,2.247971,-0.446372,0.294562,0.511107,...,+,0.658388,0.102688,+,2,0.857143,5,0.428571,5,0.428571
5,PaymentMethod_Credit_card,-0.577871,0.577871,-0.233549,0.233549,-0.249553,3.090064,-0.049553,0.404905,0.334775,...,-,0.537227,-0.270713,-,7,0.142857,6,0.285714,6,0.285714
6,PhoneService,-1.071605,1.071605,-0.433093,0.433093,0.000000,0.000000,0.000000,0.000000,0.433093,...,-,0.433093,-0.433093,-,6,0.285714,7,0.142857,7,0.142857
7,PaymentMethod_Bank_transfer,-0.480570,0.480570,-0.194224,0.194224,-1.248585,1.248585,-0.247928,0.163608,0.235126,...,-,0.316930,-0.380170,-,8,0.000000,8,0.000000,8,0.000000



Length-2 paths:


,Source,Via,Target,Weight_Source_Via,Weight_Via_Churn,Path_Signed,Path_Abs
0,Contract_Month-to-month,Tenure,Churn,-1.727900,-2.474307,4.275354,4.275354
1,MonthlyCharges,InternetService_Fiber_optic,Churn,3.583054,1.082427,3.878396,3.878396
2,Tenure,InternetService_Fiber_optic,Churn,-3.454452,1.082427,-3.739193,3.739193
3,InternetService_Fiber_optic,PhoneService,Churn,2.097761,-1.071605,-2.247971,2.247971
4,PaymentMethod_Mailed_check,InternetService_Fiber_optic,Churn,-1.756674,1.082427,-1.901472,1.901472
5,PaymentMethod_Mailed_check,PaymentMethod_Credit_card,Churn,-2.971882,-0.577871,1.717365,1.717365
6,PaymentMethod_Mailed_check,PaymentMethod_Bank_transfer,Churn,-3.305564,-0.480570,1.588555,1.588555
7,MonthlyCharges,PhoneService,Churn,1.473634,-1.071605,-1.579153,1.579153
8,PaymentMethod_Credit_card,PaymentMethod_Bank_transfer,Churn,-2.955355,-0.480570,1.420255,1.420255
9,PaymentMethod_Mailed_check,PhoneService,Churn,1.285995,-1.071605,-1.378079,1.378079



Edges in Churn subgraph:


,Source,Target,Weight,Direct_To_Churn
0,PaymentMethod_Bank_transfer,PhoneService,0.572972,False
1,PaymentMethod_Bank_transfer,InternetService_Fiber_optic,-0.586261,False
2,PaymentMethod_Bank_transfer,Churn,-0.480570,True
3,PaymentMethod_Credit_card,PhoneService,0.597515,False
4,PaymentMethod_Credit_card,InternetService_Fiber_optic,-0.951111,False
5,PaymentMethod_Credit_card,PaymentMethod_Bank_transfer,-2.955355,False
6,PaymentMethod_Credit_card,Churn,-0.577871,True
7,InternetService_Fiber_optic,PhoneService,2.097761,False
8,InternetService_Fiber_optic,Churn,1.082427,True
9,Tenure,MonthlyCharges,-1.301850,False


In [63]:
# ============================================================
# 13. EXPORT TO EXCEL
# ============================================================

#output_file = "Path_Based_Churn_Scores.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    results.to_excel(
        writer,
        sheet_name="Scores",
        index=False
    )

    paths_df.to_excel(
        writer,
        sheet_name="Length2_Paths",
        index=False
    )

    edges_df.to_excel(
        writer,
        sheet_name="Subgraph_Edges",
        index=False
    )


In [64]:
print(f"\nResults saved to: {output_file}")


Results saved to: Path_Based_Structural_Importance_DAGMA.xlsx
